In [1]:
# This notebook is aimed to generate baseline weights for the MAE to get familiar with MRI images

In [2]:
! pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 29.6 MB/s eta 0:00:00


In [3]:
"""
MAE-based Unsupervised Anomaly Detection for FCD (Focal Cortical Dysplasia)
===========================================================================
Pipeline:
  1. Train MaskedAutoEncoderViT on healthy (HC) MRI volumes only
  2. At inference, reconstruct lesion (FCD) volumes
  3. Compute residual map — lesion appears as high-error region
  4. Threshold residual to get binary lesion mask
  5. Evaluate against ground-truth masks
"""

import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tqdm
from glob import glob
from scipy import ndimage
from einops import rearrange
import torch.nn.functional as F
 
from monai.networks.nets.masked_autoencoder_vit import MaskedAutoEncoderViT
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ResizeWithPadOrCropd, EnsureTyped,
    RandFlipd, RandAffined, RandGaussianNoised, RandAdjustContrastd,
    SpatialPadd, RandSpatialCropd,
)
from monai.data import CacheDataset, DataLoader
from monai.losses import SSIMLoss

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-10 18:05:37.368393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778436337.580705      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778436337.642841      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778436338.160241      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778436338.160282      23 computation_placer.cc:1

In [4]:
# =============================================================================
# 0. CONFIG
# =============================================================================
IMG_SIZE      = 96       # spatial size for all 3 dims
PATCH_SIZE    = 8       # must divide IMG_SIZE evenly → 96/16 = 6
BATCH_SIZE    = 2        # increase if GPU allows
NUM_EPOCHS    = 2000      # minimum for ViT-scale MAE; aim for 800-1000
LR            = 1e-4     # base LR; warmup will scale it up from near-zero
WEIGHT_DECAY  = 0.005
WARMUP_EPOCHS = 50       # ~5% of total epochs
MASKING_RATIO = 0.85     # higher → harder task → better anomaly sensitivity
SAVE_PATH     = "mae_best_model.pth"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# =============================================================================
# 1. DATA LOADING  — paste your existing loading code here
# =============================================================================
data_root = '/kaggle/input/datasets/almadavidson1/fcd-database'
tsv_path  = os.path.join(data_root, "participants.tsv")
df        = pd.read_csv(tsv_path, sep='\t')
 
data_dicts = []
for idx, row in df.iterrows():
    p_id = row['participant_id']
 
    image_pattern = os.path.join(data_root, 'DATA', p_id, 'anat', f"*_T1w.nii*", f"*_T1w.nii*")
    img_files     = glob(image_pattern)
 
    mask_pattern  = os.path.join(data_root, 'DATA', p_id, 'anat', f"*-FLAIR_roi_inT1.nii*")
    mask_files   = glob(mask_pattern)
 
    if img_files:
        entry = {"image": img_files[0], "group": row.get('group', 'hc')}
        if mask_files:
            entry["mask"] = mask_files[0]
        data_dicts.append(entry)
 
hc_dict       = [d for d in data_dicts if str(d['group']).lower() == 'hc']
fcd_dict      = [d for d in data_dicts if str(d['group']).lower() == 'fcd' and 'mask' in d]
fcd_dict_clean = [d for d in fcd_dict if os.path.getsize(d["mask"]) > 0]
 
print(f"HC subjects:             {len(hc_dict)}")
print(f"FCD subjects with mask:  {len(fcd_dict_clean)}")
 
# Training on healthy only; val on remaining healthy + all FCD
train_files = hc_dict[:75]
val_hc      = hc_dict[75:]

HC subjects:             85
FCD subjects with mask:  85


In [6]:
# =============================================================================
# 2. TRANSFORMS
# =============================================================================
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    RandSpatialCropd(keys=["image"], roi_size=(IMG_SIZE, IMG_SIZE, IMG_SIZE), random_size=False),
    EnsureTyped(keys=["image"], dtype=torch.float32),
])
 
def make_infer_transforms(load_mask=False):
    keys = ["image", "mask"] if load_mask else ["image"]
    transforms = [
        LoadImaged(keys=keys),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes="RAS"),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        RandSpatialCropd(keys=["image"], roi_size=(IMG_SIZE, IMG_SIZE, IMG_SIZE), random_size=False),
        EnsureTyped(keys=keys, dtype=torch.float32),
    ]
    return Compose(transforms)
 
val_transforms     = make_infer_transforms(load_mask=False)
fcd_transforms     = make_infer_transforms(load_mask=True)

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [7]:
# =============================================================================
# 3. DATASETS & DATALOADERS
# =============================================================================
train_ds = CacheDataset(data=train_files, transform=train_transforms,
                        cache_rate=1.0, num_workers=4)
val_ds   = CacheDataset(data=val_hc,     transform=val_transforms,
                        cache_rate=1.0, num_workers=2)
fcd_ds   = CacheDataset(data=fcd_dict_clean, transform=fcd_transforms,
                        cache_rate=1.0, num_workers=2)
 
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1,          shuffle=False,
                          num_workers=2)
fcd_loader   = DataLoader(fcd_ds,   batch_size=1,          shuffle=False,
                          num_workers=2)
 
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | FCD cases: {len(fcd_ds)}")

Loading dataset: 100%|██████████| 85/85 [01:38<00:00,  1.15s/it]

Train batches: 38 | Val batches: 10 | FCD cases: 85


In [8]:
# =============================================================================
# 4. MODEL
# =============================================================================
model = MaskedAutoEncoderViT(
    spatial_dims=3,
    in_channels=1,
    img_size=(IMG_SIZE, IMG_SIZE, IMG_SIZE),
    patch_size=(PATCH_SIZE, PATCH_SIZE, PATCH_SIZE),
    masking_ratio=MASKING_RATIO,
).to(DEVICE)
 
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params / 1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Model parameters: 73.3M


In [9]:
# =============================================================================
# 5. LOSS, OPTIMIZER, SCHEDULER
# ============================================================================= 
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),   # MAE paper recommendation — 0.95 not default 0.999
)
 
# Warmup → Cosine annealing
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
 
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=1e-3,    # LR starts at LR * 1e-3, ramps up to LR
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS-WARMUP_EPOCHS,
    eta_min=1e-6,
)
scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS],
)
 

In [10]:
# =============================================================================
# 6. TRAINING HELPERS
# =============================================================================
GRID = IMG_SIZE // PATCH_SIZE   # 6
 
def patchify(x):
    """(B, 1, H, W, D) → (B, N_patches, patch_volume)"""
    return rearrange(
        x, 'b c (h p1) (w p2) (d p3) -> b (h w d) (p1 p2 p3 c)',
        p1=PATCH_SIZE, p2=PATCH_SIZE, p3=PATCH_SIZE
    )
 
def unpatchify(patches):
    """(B, N_patches, patch_volume) → (B, 1, H, W, D)"""
    return rearrange(
        patches,
        'b (h w d) (p1 p2 p3 c) -> b c (h p1) (w p2) (d p3)',
        h=GRID, w=GRID, d=GRID,
        p1=PATCH_SIZE, p2=PATCH_SIZE, p3=PATCH_SIZE, c=1
    )

In [11]:
# =============================================================================
# 7. TRAINING LOOP
# =============================================================================
best_val_loss  = float("inf")
train_losses   = []
val_losses     = []
 
# Fixed healthy volume for visual monitoring every 10 epochs
viz_volume = next(iter(val_loader))["image"][[0]].to(DEVICE)
 
epoch_bar = tqdm.tqdm(range(NUM_EPOCHS), desc="MAE Pre-training", unit="epoch")
 
for epoch in epoch_bar:
 
    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    epoch_train_loss = 0.0
    optimizer.zero_grad()
 
    for batch in train_loader:
        x = batch["image"].to(DEVICE)
 
        pred, mask   = model(x)
        x_patch = patchify(x)
        loss = F.mse_loss(x_patch[mask.bool()],pred[mask.bool()])
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()
 
        epoch_train_loss += loss.item()
 
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
 
    # ── VALIDATE ───────────────────────────────────────────────────────────
    model.eval()
    epoch_val_loss = 0.0
 
    with torch.no_grad():
        for batch in val_loader:
            x            = batch["image"].to(DEVICE)
            pred, mask   = model(x)
            x_patch      = patchify(x)
            loss         = F.mse_loss(x_patch[mask.bool()],pred[mask.bool()])
            epoch_val_loss += loss.item()
 
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
 
    scheduler.step()
 
    epoch_bar.set_postfix({
        "train": f"{avg_train_loss:.5f}",
        "val":   f"{avg_val_loss:.5f}",
        "lr":    f"{optimizer.param_groups[0]['lr']:.2e}",
    })
 
    # ── CHECKPOINT ─────────────────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "epoch":               epoch + 1,
            "model_state_dict":    model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_loss":            best_val_loss,
            "train_losses":        train_losses,
            "val_losses":          val_losses,
        }, SAVE_PATH)
        epoch_bar.write(f"  ✓ Epoch {epoch+1}: saved best model  (val={best_val_loss:.5f})")
 
    # ── VISUAL MONITORING every 10 epochs ──────────────────────────────────
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            viz_pred, viz_mask = model(viz_volume)
            viz_composed = unpatchify(viz_pred)
 
        orig_np  = viz_volume.squeeze().cpu().numpy()
        recon_np = viz_composed.squeeze().cpu().numpy()
        resid_np = np.abs(orig_np - recon_np)
        s        = orig_np.shape[2] // 2   # central axial slice
 
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].imshow(orig_np[:, :, s],  cmap="gray"); axes[0].set_title("Original (HC)")
        axes[1].imshow(recon_np[:, :, s], cmap="gray"); axes[1].set_title("Reconstruction")
        axes[2].imshow(resid_np[:, :, s], cmap="hot");  axes[2].set_title("Residual")
        for ax in axes: ax.axis("off")
        plt.suptitle(f"Epoch {epoch+1}  |  train={avg_train_loss:.4f}  val={avg_val_loss:.4f}")
        plt.tight_layout()
        plt.savefig(f"recon_epoch_{epoch+1:04d}.png", dpi=100)
        plt.close()
        epoch_bar.write(f"  📊 Epoch {epoch+1}: reconstruction plot saved")
 
print(f"\nTraining complete. Best val loss: {best_val_loss:.5f}")

MAE Pre-training:   0%|          | 1/2000 [00:14<7:47:36, 14.04s/epoch, train=1.58691, val=1.65378, lr=2.10e-06]

  ✓ Epoch 1: saved best model  (val=1.65378)


MAE Pre-training:   0%|          | 2/2000 [00:27<7:32:33, 13.59s/epoch, train=1.29871, val=1.30646, lr=4.10e-06]

  ✓ Epoch 2: saved best model  (val=1.30646)


MAE Pre-training:   0%|          | 3/2000 [00:40<7:31:00, 13.55s/epoch, train=1.11500, val=0.94142, lr=6.09e-06]

...

  📊 Epoch 1980: reconstruction plot saved


MAE Pre-training: 100%|█████████▉| 1990/2000 [7:29:24<02:19, 13.91s/epoch, train=0.15469, val=0.15177, lr=1.01e-06]

  📊 Epoch 1990: reconstruction plot saved


MAE Pre-training: 100%|██████████| 2000/2000 [7:31:41<00:00, 13.55s/epoch, train=0.18150, val=0.14241, lr=1.00e-06]

  📊 Epoch 2000: reconstruction plot saved

Training complete. Best val loss: 0.09807
